In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.linear_model import LinearRegression



master_df = pd.read_csv(
    r"C:\Users\HEMANATH\Desktop\FIFA\data\processed\clustered_players.csv"
)

master_df.shape

(17467, 47)

In [3]:
features = [
    "age",
    "matches",
    "minutes_played",
    "goals",
    "assists",
    "international_caps",
    "international_goals"
]
print(master_df.columns.tolist())

['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur', 'matches', 'goals', 'assists', 'yellow_cards', 'red_cards', 'minutes_played', 'data_of_birth', 'age', 'goal_per90', 'assists_per90', 'goals_per90', 'card_per90', 'cards_per90', 'log_market_value', 'intl_goal_ratio', 'position_group', 'goal_contribution_per90', 'cluster', 'cluster_name', 'PC1', 'PC2']


In [4]:
master_df[features].isna().sum()

age                        0
matches                    0
minutes_played             0
goals                      0
assists                    0
international_caps     10486
international_goals    10486
dtype: int64

In [5]:
master_df["international_caps"] = (
    master_df["international_caps"]
    .fillna(0)
)

master_df["international_goals"] = (
    master_df["international_goals"]
    .fillna(0)
)

In [18]:
master_df = master_df.dropna(
    subset=["market_value_in_eur"]
)

In [19]:
X = master_df[features].fillna(0)
y=master_df['market_value_in_eur']

In [20]:
master_df['market_value_in_eur'].isna().sum()

np.int64(0)

In [21]:
y_log = np.log1p(y) 
y_log.isnull().sum()

np.int64(0)

In [22]:
master_df[features].isna().sum()

age                    0
matches                0
minutes_played         0
goals                  0
assists                0
international_caps     0
international_goals    0
dtype: int64

In [23]:
master_df['market_value_in_eur'].isna().sum()

np.int64(0)

In [24]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y_log,test_size=0.2,random_state=42,shuffle=True)

In [ ]:
lr=LinearRegression()

lr.fit(X_train,y_train)

preds=lr.predict(X_test)

In [28]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

mae=mean_absolute_error(y_test,preds)
rmse=np.sqrt(mean_squared_error(y_test,preds))
r2=r2_score(y_test,preds)


print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 0.8863419773529007
RMSE: 1.1115121227276794
R2: 0.5201558014233347


In [31]:
from sklearn.ensemble import RandomForestRegressor

rf=RandomForestRegressor(n_estimators=200,random_state=42,n_jobs=-1)

rf.fit(X_train,y_train)

rf_preds=rf.predict(X_test)

In [32]:
print(
    r2_score(y_test, rf_preds)
)

0.6251915101772785


In [33]:
importance = pd.DataFrame(
    {
        "feature": X.columns,
        "importance": rf.feature_importances_
    }
).sort_values(
    "importance",
    ascending=False
)

importance

,feature,importance
5,international_caps,0.363012
0,age,0.206889
2,minutes_played,0.171587
1,matches,0.135525
4,assists,0.057133
3,goals,0.056542
6,international_goals,0.009312


In [34]:
import plotly.express as px

fig=px.bar(importance,x='importance',y='feature',orientation="h",title="feature importance")
fig.show()


In [35]:
import pickle

pickle.dump(
    rf,
    open(
        "../models/market_value_model.pkl",
        "wb"
    )
)